In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct",
    local_dir="./models/qwen2.5-0.5b-instruct",
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./models/qwen2.5-0.5b-instruct"

# Smoke test: confirm the downloaded model loads and produces SQL.
with open("schema_prompt.txt") as f:
    schema = f.read()

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float32).to("cpu")
model.eval()

messages = [
    {"role": "system", "content": "You are a SQL expert. Given a database schema and a question, write the correct SQL query."},
    {"role": "user",   "content": f"Schema:{schema}\nQuestion: How many customers are in the 'vip' segment?"},
]
tokenized = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
)
input_ids = tokenized if isinstance(tokenized, torch.Tensor) else tokenized["input_ids"]

with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        attention_mask=torch.ones_like(input_ids),
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip())